In [2]:
import os
import streamlit as st
import pandas as pd
import sys
from pathlib import Path
import numpy as np
from datetime import datetime, timedelta

# Add the parent directory to sys.path so 'dashboard' can be imported if it's a local package
current_dir = Path(os.getcwd())
parent_dir = current_dir.parent
if str(parent_dir) not in sys.path:
	sys.path.insert(0, str(parent_dir))
 
from dashboard.database.db_connection import DatabaseConnection

# Get database path

db_path = parent_dir / "dashboard" / "database" / "bikes.db"
print(db_path)
# Initialize database connection
db = DatabaseConnection(str(db_path))


INFO:dashboard.database.db_connection:Connected to database: /Users/ansen/Documents/GitHub/applied-data-science/dashboard/database/bikes.db


/Users/ansen/Documents/GitHub/applied-data-science/dashboard/database/bikes.db


In [3]:
available_dates_sql = """
SELECT DISTINCT selected_date as pred_date
FROM pred_bike_rentals_24h
ORDER BY pred_date DESC
LIMIT 30
"""
available_dates_result = db.execute_query(available_dates_sql)
print(available_dates_result)

INFO:dashboard.database.db_connection:Query executed successfully


[('2018-11-30',), ('2018-11-29',), ('2018-11-28',), ('2018-11-27',), ('2018-11-26',), ('2018-11-25',), ('2018-11-24',), ('2018-11-23',), ('2018-11-22',), ('2018-11-21',), ('2018-11-20',)]


In [6]:
# Get dates for selection
selected_date = datetime(2018, 11, 25)
selected_hour = 10

# Prepare data
selected_date_str = selected_date.strftime('%Y-%m-%d')
selected_datetime = datetime(selected_date.year, selected_date.month, selected_date.day, selected_hour, 0, 0)
week_ago = selected_datetime - timedelta(days=7)

print("Selected datetime:", selected_datetime)
print("Week ago:", week_ago)

Selected datetime: 2018-11-25 10:00:00
Week ago: 2018-11-18 10:00:00


In [7]:

# Query historical data (last 7 days)
hist_sql = f"""
SELECT datetime(date || ' ' || printf('%02d:00:00', hour)) as datetime, 
        rented_bike_count as bikes
FROM stg_bike_rentals_hourly
WHERE datetime(date || ' ' || printf('%02d:00:00', hour)) >= '{week_ago}' 
AND datetime(date || ' ' || printf('%02d:00:00', hour)) <= '{selected_datetime}'
ORDER BY datetime
"""
hist_results = db.execute_query(hist_sql)

# Query 24-hour predictions (generated at selected time)
pred_24h_sql = f"""
SELECT prediction_datetime, predicted_bikes as prediction
FROM pred_bike_rentals_24h 
WHERE selected_date = '{selected_date_str}' 
AND selected_hour = {selected_hour}
ORDER BY prediction_datetime
"""
pred_24h_results = db.execute_query(pred_24h_sql)

# Query 3-day predictions (generated at selected time)
pred_3d_sql = f"""
SELECT prediction_datetime, predicted_bikes as prediction
FROM pred_bike_rentals_3d 
WHERE selected_date = '{selected_date_str}'
AND selected_hour = {selected_hour}
ORDER BY prediction_datetime
"""
pred_3d_results = db.execute_query(pred_3d_sql)

INFO:dashboard.database.db_connection:Query executed successfully
INFO:dashboard.database.db_connection:Query executed successfully
INFO:dashboard.database.db_connection:Query executed successfully


In [8]:
print(hist_results)
print(pred_24h_results)
print(pred_3d_results)

[('2018-11-18 10:00:00', 583), ('2018-11-18 11:00:00', 645), ('2018-11-18 12:00:00', 776), ('2018-11-18 13:00:00', 905), ('2018-11-18 14:00:00', 1054), ('2018-11-18 15:00:00', 1047), ('2018-11-18 16:00:00', 1026), ('2018-11-18 17:00:00', 1039), ('2018-11-18 18:00:00', 902), ('2018-11-18 19:00:00', 759), ('2018-11-18 20:00:00', 719), ('2018-11-18 21:00:00', 682), ('2018-11-18 22:00:00', 632), ('2018-11-18 23:00:00', 532), ('2018-11-19 00:00:00', 457), ('2018-11-19 01:00:00', 277), ('2018-11-19 02:00:00', 190), ('2018-11-19 03:00:00', 133), ('2018-11-19 04:00:00', 88), ('2018-11-19 05:00:00', 167), ('2018-11-19 06:00:00', 418), ('2018-11-19 07:00:00', 991), ('2018-11-19 08:00:00', 1751), ('2018-11-19 09:00:00', 889), ('2018-11-19 10:00:00', 651), ('2018-11-19 11:00:00', 745), ('2018-11-19 12:00:00', 785), ('2018-11-19 13:00:00', 785), ('2018-11-19 14:00:00', 793), ('2018-11-19 15:00:00', 839), ('2018-11-19 16:00:00', 968), ('2018-11-19 17:00:00', 1233), ('2018-11-19 18:00:00', 1904), ('2

In [9]:
# Prepare DataFrames
hist_data = pd.DataFrame()
if hist_results:
    hist_data = pd.DataFrame(hist_results, columns=['datetime', 'bikes'])
    hist_data['datetime'] = pd.to_datetime(hist_data['datetime'])
    hist_data['bikes'] = pd.to_numeric(hist_data['bikes'], errors='coerce')

pred_24h_data = pd.DataFrame()
if pred_24h_results:
    pred_24h_data = pd.DataFrame(pred_24h_results, columns=['datetime', 'prediction'])
    pred_24h_data['datetime'] = pd.to_datetime(pred_24h_data['datetime'])
    pred_24h_data['prediction'] = pd.to_numeric(pred_24h_data['prediction'], errors='coerce')

pred_3d_data = pd.DataFrame()
if pred_3d_results:
    pred_3d_data = pd.DataFrame(pred_3d_results, columns=['datetime', 'prediction'])
    pred_3d_data['datetime'] = pd.to_datetime(pred_3d_data['datetime'])
    pred_3d_data['prediction'] = pd.to_numeric(pred_3d_data['prediction'], errors='coerce')
        

In [11]:
print("Historical Results:", hist_data.tail())
print("24-Hour Predictions:", pred_24h_data.head())
print("3-Day Predictions:", pred_3d_data.head())

Historical Results:                datetime  bikes
164 2018-11-25 06:00:00     75
165 2018-11-25 07:00:00    142
166 2018-11-25 08:00:00    250
167 2018-11-25 09:00:00    355
168 2018-11-25 10:00:00    430
24-Hour Predictions:              datetime  prediction
0 2018-11-25 11:00:00  548.260920
1 2018-11-25 12:00:00  657.196918
2 2018-11-25 13:00:00  760.138718
3 2018-11-25 14:00:00  873.645089
4 2018-11-25 15:00:00  943.879878
3-Day Predictions:              datetime  prediction
0 2018-11-25 11:00:00  526.164450
1 2018-11-25 12:00:00  603.852210
2 2018-11-25 13:00:00  701.468756
3 2018-11-25 14:00:00  790.193315
4 2018-11-25 15:00:00  877.635180


In [12]:
import plotly.graph_objects as go
# Create combined plot
fig = go.Figure()

# Add historical data
if not hist_data.empty:
    fig.add_trace(go.Scatter(
        x=hist_data['datetime'],
        y=hist_data['bikes'],
        name='Historical Data (Last 7 Days)',
        mode='lines',
        line=dict(color='#4ecdc4', width=2),
        hovertemplate='<b>Historical</b><br>%{x|%Y-%m-%d %H:%M}<br>%{y:.0f} bikes<extra></extra>'
    ))

# Add 3-day predictions
if not pred_3d_data.empty:
    fig.add_trace(go.Scatter(
        x=pred_3d_data['datetime'],
        y=pred_3d_data['prediction'],
        name='3-Day Predictions',
        mode='lines',
        line=dict(color='#95e1d3', width=2.5, dash='dot'),
        hovertemplate='<b>3d Forecast</b><br>%{x|%Y-%m-%d %H:%M}<br>%{y:.0f} bikes<extra></extra>'
    ))

# Add 24-hour predictions
if not pred_24h_data.empty:
    fig.add_trace(go.Scatter(
        x=pred_24h_data['datetime'],
        y=pred_24h_data['prediction'],
        name='24-Hour Predictions',
        mode='lines+markers',
        line=dict(color='#ff6b6b', width=2.5, dash='dash'),
        marker=dict(size=4, color='#ff6b6b'),
        hovertemplate='<b>24h Forecast</b><br>%{x|%Y-%m-%d %H:%M}<br>%{y:.0f} bikes<extra></extra>'
    ))

In [ ]:
selected_unix = int(selected_datetime.timestamp()) * 1000


In [14]:
# Add vertical line for selected datetime
selected_unix = int(selected_datetime.timestamp()) * 1000

fig.add_vline(
    x=selected_unix,
    line_color="orange",
    line_width=2,
    annotation_text=f"Selected: {selected_datetime_str}",
    annotation_position="top left",
    annotation_font_size=11,
    annotation_font_color="orange"
)

# Update layout
fig.update_layout(
    title=f"Bike Rental Predictions (24h & 3d) vs Historical Data - {selected_date.strftime('%Y-%m-%d')} {selected_hour:02d}:00",
    xaxis_title="Date & Time",
    yaxis_title="Bikes Rented",
    hovermode='x unified',
    height=600,
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(b=100)
)


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': '<b>Historical</b><br>%{x|%Y-%m-%d %H:%M}<br>%{y:.0f} bikes<extra></extra>',
              'line': {'color': '#4ecdc4', 'width': 2},
              'mode': 'lines',
              'name': 'Historical Data (Last 7 Days)',
              'type': 'scatter',
              'x': array(['2018-11-18T10:00:00.000000000', '2018-11-18T11:00:00.000000000',
                          '2018-11-18T12:00:00.000000000', '2018-11-18T13:00:00.000000000',
                          '2018-11-18T14:00:00.000000000', '2018-11-18T15:00:00.000000000',
                          '2018-11-18T16:00:00.000000000', '2018-11-18T17:00:00.000000000',
                          '2018-11-18T18:00:00.000000000', '2018-11-18T19:00:00.000000000',
                          '2018-11-18T20:00:00.000000000', '2018-11-18T21:00:00.000000000',
                          '2018-11-18T22:00:00.000000000', '2018-11-18T23:00:00.000000000',
                          '2018-11-19T00:00:00.000000000', '2018-11-19T01:00:00.000000000',
                          '2018-11-19T02:00:00.000000000', '2018-11-19T03:00:00.000000000',
                          '2018-11-19T04:00:00.000000000', '2018-11-19T05:00:00.000000000',
                          '2018-11-19T06:00:00.000000000', '2018-11-19T07:00:00.000000000',
                          '2018-11-19T08:00:00.000000000', '2018-11-19T09:00:00.000000000',
                          '2018-11-19T10:00:00.000000000', '2018-11-19T11:00:00.000000000',
                          '2018-11-19T12:00:00.000000000', '2018-11-19T13:00:00.000000000',
                          '2018-11-19T14:00:00.000000000', '2018-11-19T15:00:00.000000000',
                          '2018-11-19T16:00:00.000000000', '2018-11-19T17:00:00.000000000',
                          '2018-11-19T18:00:00.000000000', '2018-11-19T19:00:00.000000000',
                          '2018-11-19T20:00:00.000000000', '2018-11-19T21:00:00.000000000',
                          '2018-11-19T22:00:00.000000000', '2018-11-19T23:00:00.000000000',
                          '2018-11-20T00:00:00.000000000', '2018-11-20T01:00:00.000000000',
                          '2018-11-20T02:00:00.000000000', '2018-11-20T03:00:00.000000000',
                          '2018-11-20T04:00:00.000000000', '2018-11-20T05:00:00.000000000',
                          '2018-11-20T06:00:00.000000000', '2018-11-20T07:00:00.000000000',
                          '2018-11-20T08:00:00.000000000', '2018-11-20T09:00:00.000000000',
                          '2018-11-20T10:00:00.000000000', '2018-11-20T11:00:00.000000000',
                          '2018-11-20T12:00:00.000000000', '2018-11-20T13:00:00.000000000',
                          '2018-11-20T14:00:00.000000000', '2018-11-20T15:00:00.000000000',
                          '2018-11-20T16:00:00.000000000', '2018-11-20T17:00:00.000000000',
                          '2018-11-20T18:00:00.000000000', '2018-11-20T19:00:00.000000000',
                          '2018-11-20T20:00:00.000000000', '2018-11-20T21:00:00.000000000',
                          '2018-11-20T22:00:00.000000000', '2018-11-20T23:00:00.000000000',
                          '2018-11-21T00:00:00.000000000', '2018-11-21T01:00:00.000000000',
                          '2018-11-21T02:00:00.000000000', '2018-11-21T03:00:00.000000000',
                          '2018-11-21T04:00:00.000000000', '2018-11-21T05:00:00.000000000',
                          '2018-11-21T06:00:00.000000000', '2018-11-21T07:00:00.000000000',
                          '2018-11-21T08:00:00.000000000', '2018-11-21T09:00:00.000000000',
                          '2018-11-21T10:00:00.000000000', '2018-11-21T11:00:00.000000000',
                          '2018-11-21T12:00:00.000000000', '2018-11-21T13:00:00.000000000',
                          '2018-11-21T14:00:00.000000000', '2018-11-21T15:00:00.000000000',
                          '2018-11-21T16:00:00.000000000', '2018-11-21T17:00:00.000000000',
                          '2